# E-commerce Sales — Exploratory Data Analysis

A walk-through of the full analyst workflow on 3 years of synthetic online-retail orders:

1. Load the raw (messy) data
2. Assess data quality
3. Clean & prepare
4. Explore: trends, categories, regions, discounts
5. State the key business insights

> Reusable logic lives in `../src/`; this notebook is the narrative.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import load_raw, clean
from src import analysis

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Load the raw data

If this fails, run `python data/generate_data.py` from the project root first.

In [ ]:
raw = load_raw()
print(raw.shape)
raw.head()

## 2. Data-quality assessment

Before any analysis, understand what's broken: missing values, duplicates, invalid entries.

In [ ]:
print('Missing values:')
print(raw.isna().sum())
print(f'\nDuplicate rows: {raw.duplicated().sum()}')
print(f'Negative quantities: {(raw["quantity"] < 0).sum()}')

## 3. Clean & prepare

The `clean()` pipeline drops duplicates, standardizes text, fixes bad quantities,
imputes missing values, and adds derived columns (`order_month`, `profit_margin`).

In [ ]:
df = clean(raw)
print(f'Rows: {len(raw):,} -> {len(df):,} after cleaning')
print('Remaining missing values:', df.isna().sum().sum())
df.head()

## 4. Exploration

### 4.1 Headline KPIs

In [ ]:
k = analysis.kpis(df)
for name, val in k.items():
    print(f'{name:22s}: {val:,.2f}')

### 4.2 Sales trend over time

In [ ]:
trend = analysis.sales_over_time(df, freq='M')
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(trend['order_date'], trend['sales'], marker='o', label='Sales')
ax.plot(trend['order_date'], trend['profit'], marker='o', label='Profit')
ax.set_title('Monthly sales and profit')
ax.set_ylabel('Amount ($)')
ax.legend()
plt.tight_layout()

### 4.3 Sales by category and region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
analysis.sales_by(df, 'category').plot.bar(x='category', y='sales', ax=axes[0], legend=False, title='Sales by category')
analysis.sales_by(df, 'region').plot.bar(x='region', y='sales', ax=axes[1], legend=False, title='Sales by region', color='seagreen')
for ax in axes:
    ax.set_ylabel('Sales ($)')
plt.tight_layout()

### 4.4 Does discounting pay off?

In [ ]:
disc = analysis.discount_vs_profit(df)
display(disc)
ax = disc.plot.bar(x='discount_band', y='avg_margin', legend=False, color='tomato', title='Avg profit margin by discount band')
ax.set_ylabel('Avg profit margin')
plt.tight_layout()

## 5. Key insights

- **Seasonality:** sales rise noticeably in Q4 each year (holiday demand).
- **Category mix:** Technology drives the most revenue; Office Supplies has high volume but low value.
- **Discounting erodes profit:** average margin falls sharply beyond ~20% discount, with the deepest band near break-even.
- **Customer concentration:** a small set of customers accounts for a large share of sales (worth a retention focus).

**Recommendation:** cap routine discounts around 20%, reserve deeper cuts for clearance, and
build a loyalty program around the top-spending customer segment.